# MLP (red neuronal ligera) con validacion cruzada y metricas completas

Replica el flujo de 07_MODEL_LIGHTGBM_V4.ipynb (variantes, balanceo, CV y reportes CSV/PDF) pero usando un perceptron multicapa liviano para no saturar CPU con ~57 variantes.

In [1]:
import os
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc, accuracy_score, precision_recall_curve, average_precision_score
)
from matplotlib.backends.backend_pdf import PdfPages
from datetime import datetime
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN
from sklearn.model_selection import StratifiedKFold, cross_val_score


In [2]:
# === CONFIGURACION ===
INPUT_PATH = r'c:/Users/Gonzalo/Documents/GitHub/TesisAustral/Archivos_tesis/intermediate/selected/04_2025_08_23'
BASE_OUTPUT_PATH = r'c:/Users/Gonzalo/Documents/GitHub/TesisAustral/Archivos_tesis/models'
MODEL_NAME = 'MLP'
INTENTO = 1  # Cambiar si generas una nueva version
N_SPLITS = 3  # Igual que LightGBM pero con red liviana
MAX_ITER = 60  # Epocas cortas + early stopping para no saturar
LEARNING_RATE_INIT = 0.002

fecha_actual = datetime.today().strftime('%Y-%m-%d')
OUTPUT_PATH = os.path.join(BASE_OUTPUT_PATH, f'{MODEL_NAME}_{INTENTO:02d}_{fecha_actual}')
os.makedirs(OUTPUT_PATH, exist_ok=True)

BALANCE_METHODS = {
    'NONE': None,
    'SMOTE': SMOTE(random_state=42),
    'UNDER': RandomUnderSampler(random_state=42),
    'SMOTEENN': SMOTEENN(random_state=42)
}

metricas_totales = []

# Detectar variantes de datasets
variantes_X = sorted([f for f in os.listdir(INPUT_PATH) if f.startswith('X_train')])
print(f'Se detectaron {len(variantes_X)} variantes de X_train_* en {INPUT_PATH}')


def build_mlp_pipeline(num_classes: int):
    '''MLP sencillo con escalado previo, pensado para muchas variantes.'''
    hidden = (96, 48) if num_classes > 2 else (64, 32)
    mlp = MLPClassifier(
        hidden_layer_sizes=hidden,
        activation='relu',
        solver='adam',
        learning_rate_init=LEARNING_RATE_INIT,
        max_iter=MAX_ITER,
        batch_size=128,
        alpha=1e-4,
        early_stopping=True,
        n_iter_no_change=6,
        validation_fraction=0.12,
        shuffle=True,
        random_state=42,
        verbose=False,
    )
    # with_mean=False evita problemas si algun set es esparso; mantiene liviano
    return make_pipeline(StandardScaler(with_mean=False), mlp)


Se detectaron 59 variantes de X_train_* en c:/Users/Gonzalo/Documents/GitHub/TesisAustral/Archivos_tesis/intermediate/selected/04_2025_08_23


In [3]:
for balance_name, balancer in BALANCE_METHODS.items():
    print(f"=== Balanceo: {balance_name} ===")
    output_subdir = os.path.join(OUTPUT_PATH, balance_name)
    os.makedirs(output_subdir, exist_ok=True)

    for x_file in variantes_X:
        base_name = x_file.replace('X_train_', '').replace('.parquet', '')
        try:
            print(f' Procesando: {base_name}')

            # Cargar datos
            X_train = pd.read_parquet(os.path.join(INPUT_PATH, f'X_train_{base_name}.parquet'))
            X_test = pd.read_parquet(os.path.join(INPUT_PATH, f'X_test_{base_name}.parquet'))
            y_train = pd.read_parquet(os.path.join(INPUT_PATH, f'y_train_{base_name}.parquet')).squeeze()
            y_test = pd.read_parquet(os.path.join(INPUT_PATH, f'y_test_{base_name}.parquet')).squeeze()

            # Ajustar etiquetas para que arranquen en 0
            y_train = y_train - y_train.min()
            y_test = y_test - y_test.min()

            # Balanceo solo en train
            if balancer is not None:
                X_train_bal, y_train_bal = balancer.fit_resample(X_train, y_train)
            else:
                X_train_bal, y_train_bal = X_train, y_train

            num_classes = int(pd.Series(y_train_bal).nunique())
            if num_classes < 2:
                print('  Dataset con una sola clase; se omite.')
                continue

            # Validacion cruzada rapida sobre el train balanceado
            cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
            model_cv = build_mlp_pipeline(num_classes)
            cv_scores = cross_val_score(model_cv, X_train_bal, y_train_bal, cv=cv, scoring='f1_macro')
            cv_f1_mean = float(np.mean(cv_scores))
            cv_f1_std = float(np.std(cv_scores))

            # Entrenamiento final
            start = time.time()
            model = build_mlp_pipeline(num_classes)
            model.fit(X_train_bal, y_train_bal)
            tiempo = (time.time() - start) / 60

            # Prediccion
            y_pred_train = model.predict(X_train_bal)
            y_pred_test = model.predict(X_test)
            y_proba = model.predict_proba(X_test)
            class_order = model.named_steps['mlp'].classes_

            # Reporte completo
            report_test = pd.DataFrame(classification_report(y_test, y_pred_test, output_dict=True)).T
            report_train = pd.DataFrame(classification_report(y_train_bal, y_pred_train, output_dict=True)).T
            report_test['set'] = 'test'
            report_train['set'] = 'train'
            full_report = pd.concat([report_train, report_test])
            full_report['tiempo_min'] = tiempo

            # Guardar CSV de metricas por clase
            full_report.to_csv(os.path.join(output_subdir, f'metricas_{base_name}.csv'))

            # Matriz de confusion
            cm = confusion_matrix(y_test, y_pred_test)

            # Ajustes para curvas ROC/PR en binario
            y_bin = label_binarize(y_test, classes=class_order)
            if y_bin.shape[1] == 1:
                y_bin = np.hstack([1 - y_bin, y_bin])
            proba_for_curves = y_proba
            if proba_for_curves.shape[1] == 1:
                proba_for_curves = np.hstack([1 - y_proba, y_proba])

            # Guardar PDF con visualizaciones
            with PdfPages(os.path.join(output_subdir, f'reporte_{base_name}.pdf')) as pdf:
                # Confusion Matrix
                ConfusionMatrixDisplay(confusion_matrix=cm).plot(cmap='Blues')
                plt.title('Matriz de Confusion')
                pdf.savefig(); plt.close()

                # ROC por clase
                plt.figure()
                for i, clase in enumerate(class_order):
                    fpr, tpr, _ = roc_curve(y_bin[:, i], proba_for_curves[:, i])
                    roc_auc = auc(fpr, tpr)
                    plt.plot(fpr, tpr, label=f'Clase {clase} (AUC={roc_auc:.2f})')
                plt.plot([0, 1], [0, 1], 'k--')
                plt.title('Curvas ROC por clase')
                plt.xlabel('FPR')
                plt.ylabel('TPR')
                plt.legend()
                pdf.savefig(); plt.close()

                # PR Curve por clase
                plt.figure()
                for i, clase in enumerate(class_order):
                    precision, recall, _ = precision_recall_curve(y_bin[:, i], proba_for_curves[:, i])
                    pr_auc = average_precision_score(y_bin[:, i], proba_for_curves[:, i])
                    plt.plot(recall, precision, label=f'Clase {clase} (PR AUC={pr_auc:.2f})')
                plt.title('Curvas Precision-Recall por clase')
                plt.xlabel('Recall')
                plt.ylabel('Precision')
                plt.legend()
                pdf.savefig(); plt.close()

            # Resumen final para comparabilidad
            resumen = {
                'modelo': base_name,
                'balanceo': balance_name,
                'accuracy_test': accuracy_score(y_test, y_pred_test),
                'macro_f1_test': report_test.loc['macro avg', 'f1-score'],
                'weighted_f1_test': report_test.loc['weighted avg', 'f1-score'],
                'accuracy_train': accuracy_score(y_train_bal, y_pred_train),
                'macro_f1_train': report_train.loc['macro avg', 'f1-score'],
                'weighted_f1_train': report_train.loc['weighted avg', 'f1-score'],
                'tiempo_min': tiempo,
                'sobreajuste_macro_f1': report_train.loc['macro avg', 'f1-score'] - report_test.loc['macro avg', 'f1-score'],
                'cv_macro_f1_mean': cv_f1_mean,
                'cv_macro_f1_std': cv_f1_std
            }
            for clase in report_test.index:
                if clase not in ['accuracy', 'macro avg', 'weighted avg']:
                    resumen[f'f1_{clase}_test'] = report_test.loc[clase, 'f1-score']
                    resumen[f'recall_{clase}_test'] = report_test.loc[clase, 'recall']
                    resumen[f'precision_{clase}_test'] = report_test.loc[clase, 'precision']
                    resumen[f'f1_{clase}_train'] = report_train.loc[clase, 'f1-score']
                    resumen[f'recall_{clase}_train'] = report_train.loc[clase, 'recall']
                    resumen[f'precision_{clase}_train'] = report_train.loc[clase, 'precision']
            metricas_totales.append(resumen)

            print(f" {base_name} [{balance_name}]: F1_clase_1_test={resumen.get('f1_1_test', 0):.3f}, Recall_clase_1_test={resumen.get('recall_1_test', 0):.3f}, Tiempo={tiempo:.2f}min")

        except Exception as e:
            print(f" Error en {base_name} con balanceo {balance_name}: {e}")


=== Balanceo: NONE ===
 Procesando: MaxAbs_FE_ALL
 Error en MaxAbs_FE_ALL con balanceo NONE: 'mlp'
 Procesando: MaxAbs_FE_ANOVA
 Error en MaxAbs_FE_ANOVA con balanceo NONE: 'mlp'
 Procesando: MaxAbs_FE_MI
 Error en MaxAbs_FE_MI con balanceo NONE: 'mlp'
 Procesando: MaxAbs_FE_PCA30
 Error en MaxAbs_FE_PCA30 con balanceo NONE: 'mlp'
 Procesando: MaxAbs_FE_PCAopt
 Error en MaxAbs_FE_PCAopt con balanceo NONE: 'mlp'
 Procesando: MaxAbs_FE_VAR
 Error en MaxAbs_FE_VAR con balanceo NONE: 'mlp'
 Procesando: MaxAbs_ORIGINAL_ALL
 Error en MaxAbs_ORIGINAL_ALL con balanceo NONE: 'mlp'
 Procesando: MaxAbs_ORIGINAL_ANOVA
 Error en MaxAbs_ORIGINAL_ANOVA con balanceo NONE: 'mlp'
 Procesando: MaxAbs_ORIGINAL_MI
 Error en MaxAbs_ORIGINAL_MI con balanceo NONE: 'mlp'
 Procesando: MaxAbs_ORIGINAL_PCA30
 Error en MaxAbs_ORIGINAL_PCA30 con balanceo NONE: 'mlp'
 Procesando: MaxAbs_ORIGINAL_PCAopt
 Error en MaxAbs_ORIGINAL_PCAopt con balanceo NONE: 'mlp'
 Procesando: MaxAbs_ORIGINAL_VAR
 Error en MaxAbs_ORIGINA

c:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (60) reached and the optimization hasn't converged yet.
  warnings.warn(


 Error en MaxAbs_ORIGINAL_PCAopt con balanceo SMOTEENN: 'mlp'
 Procesando: MaxAbs_ORIGINAL_VAR
 Error en MaxAbs_ORIGINAL_VAR con balanceo SMOTEENN: 'mlp'
 Procesando: MinMax_FE_ALL
 Error en MinMax_FE_ALL con balanceo SMOTEENN: 'mlp'
 Procesando: MinMax_FE_ANOVA
 Error en MinMax_FE_ANOVA con balanceo SMOTEENN: 'mlp'
 Procesando: MinMax_FE_MI
 Error en MinMax_FE_MI con balanceo SMOTEENN: 'mlp'
 Procesando: MinMax_FE_PCA30
 Error en MinMax_FE_PCA30 con balanceo SMOTEENN: 'mlp'
 Procesando: MinMax_FE_PCAopt
 Error en MinMax_FE_PCAopt con balanceo SMOTEENN: 'mlp'
 Procesando: MinMax_FE_VAR
 Error en MinMax_FE_VAR con balanceo SMOTEENN: 'mlp'
 Procesando: MinMax_ORIGINAL_ALL
 Error en MinMax_ORIGINAL_ALL con balanceo SMOTEENN: 'mlp'
 Procesando: MinMax_ORIGINAL_ANOVA
 Error en MinMax_ORIGINAL_ANOVA con balanceo SMOTEENN: 'mlp'
 Procesando: MinMax_ORIGINAL_CHI2
 Error en MinMax_ORIGINAL_CHI2 con balanceo SMOTEENN: 'mlp'
 Procesando: MinMax_ORIGINAL_MI
 Error en MinMax_ORIGINAL_MI con balance

c:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (60) reached and the optimization hasn't converged yet.
  warnings.warn(


 Error en MinMax_ORIGINAL_PCAopt con balanceo SMOTEENN: 'mlp'
 Procesando: MinMax_ORIGINAL_VAR
 Error en MinMax_ORIGINAL_VAR con balanceo SMOTEENN: 'mlp'
 Procesando: NoNorm_FE_ALL
 Error en NoNorm_FE_ALL con balanceo SMOTEENN: Input X contains NaN.
MLPClassifier does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values
 Procesando: NoNorm_FE_ANOVA
 Error en NoNorm_FE_ANOVA con balanceo SMOTEENN: 'mlp'
 Procesando: NoNorm_FE_MI
 Error en NoNorm_FE_MI con

...

 Error en Standard_FE_ANOVA con balanceo SMOTEENN: 'mlp'
 Procesando: Standard_FE_MI


c:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (60) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (60) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (60) reached and the optimization hasn't converged yet.
  warnings.warn(


 Error en Standard_FE_MI con balanceo SMOTEENN: 'mlp'
 Procesando: Standard_FE_PCA30
 Error en Standard_FE_PCA30 con balanceo SMOTEENN: 'mlp'
 Procesando: Standard_FE_PCAopt
 Error en Standard_FE_PCAopt con balanceo SMOTEENN: 'mlp'
 Procesando: Standard_FE_VAR
 Error en Standard_FE_VAR con balanceo SMOTEENN: 'mlp'
 Procesando: Standard_ORIGINAL_ALL
 Error en Standard_ORIGINAL_ALL con balanceo SMOTEENN: 'mlp'
 Procesando: Standard_ORIGINAL_ANOVA
 Error en Standard_ORIGINAL_ANOVA con balanceo SMOTEENN: 'mlp'
 Procesando: Standard_ORIGINAL_MI
 Error en Standard_ORIGINAL_MI con balanceo SMOTEENN: 'mlp'
 Procesando: Standard_ORIGINAL_PCA30
 Error en Standard_ORIGINAL_PCA30 con balanceo SMOTEENN: 'mlp'
 Procesando: Standard_ORIGINAL_PCAopt


c:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (60) reached and the optimization hasn't converged yet.
  warnings.warn(


 Error en Standard_ORIGINAL_PCAopt con balanceo SMOTEENN: 'mlp'
 Procesando: Standard_ORIGINAL_VAR
 Error en Standard_ORIGINAL_VAR con balanceo SMOTEENN: 'mlp'


In [4]:
# Consolidar todas las metricas en un CSV unico
if metricas_totales:
    df_metricas = pd.DataFrame(metricas_totales)
    df_metricas.to_csv(os.path.join(OUTPUT_PATH, 'metricas_totales.csv'), index=False)
    print(f'Se guardo metricas_totales.csv con {len(df_metricas)} filas en {OUTPUT_PATH}')
    display(df_metricas.head())
else:
    print('No se genero ninguna metrica; revisa los logs de arriba.')


No se genero ninguna metrica; revisa los logs de arriba.
